In [ ]:
%pip install scikit-learn --quiet

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.append(r"E:\Projet_Alzheimer\configs")
from config_projet import C

# Dernier dossier preparation_modele_* produit par le notebook 07.
dossiers = sorted(C.COHORTE.glob("preparation_modele_*"))
if not dossiers:
    raise FileNotFoundError("Lance d'abord le notebook 07 (dossier preparation_modele_* absent).")
PREP = dossiers[-1]
print("Donnees :", PREP.name)

jeu = pd.read_csv(PREP / "jeu_modelisation_359.csv")
print("Jeu charge :", jeu.shape)

In [ ]:
# Variables connues a la visite de depart (rien qui vienne du futur).
# On EXCLUT volontairement dx_fin, dx_max, label_fin, suivi_jours et les dates
# (informations posterieures qui trahiraient la cible), ainsi que MOCA (vide)
# et les biomarqueurs plasma (~10% remplis).
FEATURES = [
    "sexe", "education", "age_reference", "APOE_e4",
    "CDR_global", "CDR_SB", "MMSE_debut", "ADAS13", "FAQ_debut",
    "ABETA42", "TAU", "PTAU",
    "dx_debut",   # diagnostic de depart : legitime, connu au moment de predire
]

# Securite : ignorer une variable entierement vide (ex. age si non calcule).
vides = [c for c in FEATURES if jeu[c].notna().sum() == 0]
if vides:
    print("Colonnes ignorees car entierement vides :", vides)
FEATURES = [c for c in FEATURES if c not in vides]
print("Variables utilisees :", FEATURES)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

SCORES = ["roc_auc", "balanced_accuracy", "f1", "accuracy"]
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def modeles():
    num = SimpleImputer(strategy="median")  # remplace les valeurs manquantes
    return {
        "Hasard (Dummy)": Pipeline([
            ("imp", num), ("clf", DummyClassifier(strategy="most_frequent")),
        ]),
        "Regression logistique": Pipeline([
            ("imp", num), ("sc", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
        ]),
        "Random forest": Pipeline([
            ("imp", num),
            ("clf", RandomForestClassifier(
                n_estimators=300, class_weight="balanced", random_state=0)),
        ]),
    }

def evaluer(X, y, titre):
    print(f"\n=== {titre} ===")
    print("Participants :", len(y), "| positifs :", int(y.sum()),
          f"({y.mean() * 100:.1f}%)")
    lignes = []
    for nom, modele in modeles().items():
        res = cross_validate(modele, X, y, cv=CV, scoring=SCORES)
        lignes.append({
            "modele": nom,
            **{s: f"{res['test_' + s].mean():.3f} ± {res['test_' + s].std():.3f}"
               for s in SCORES},
        })
    tableau = pd.DataFrame(lignes).set_index("modele")
    display(tableau)
    return tableau

In [ ]:
# Tache 1 : progression (declin du diagnostic), sur les 359 participants.
jeu1 = jeu.dropna(subset=["progression"]).copy()
X1 = jeu1[FEATURES]
y1 = jeu1["progression"].astype(int)
resultat_progression = evaluer(X1, y1, "Tache 1 — Progression (359 participants)")

In [ ]:
# Tache 2 : conversion MCI -> demence, sur les participants MCI au depart.
jeu2 = jeu.loc[jeu["conversion_MCI_demence"].notna()].copy()
# dx_debut est constant ici (tous MCI) : on le retire des variables.
FEATURES2 = [c for c in FEATURES if c != "dx_debut"]
X2 = jeu2[FEATURES2]
y2 = jeu2["conversion_MCI_demence"].astype(int)
resultat_conversion = evaluer(X2, y2, "Tache 2 — Conversion MCI->demence")

In [ ]:
# Importance des variables (indicatif) : random forest ajustee sur tout le jeu.
def importances(X, y):
    Xi = SimpleImputer(strategy="median").fit_transform(X)
    rf = RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=0
    ).fit(Xi, y)
    return (pd.Series(rf.feature_importances_, index=X.columns)
            .sort_values(ascending=False))

print("Tache 1 — importance des variables :")
display(importances(X1, y1).to_frame("importance").round(3))
print("Tache 2 — importance des variables :")
display(importances(X2, y2).to_frame("importance").round(3))

In [ ]:
# Enregistrer les tableaux de resultats.
SORTIE = C.sortie("baseline_tabulaire")
resultat_progression.to_csv(SORTIE / "resultats_progression.csv", encoding="utf-8-sig")
resultat_conversion.to_csv(SORTIE / "resultats_conversion.csv", encoding="utf-8-sig")
print("Resultats enregistres dans :", SORTIE)